In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # E7.4 · Productos Gold — Trust & Safety
# MAGIC
# MAGIC ## Objetivo
# MAGIC
# MAGIC Construir productos analíticos certificados a partir de:
# MAGIC
# MAGIC - Silver conformada
# MAGIC - Evaluaciones Trust & Safety
# MAGIC - Catálogo/versiones de reglas
# MAGIC - Ejecuciones de evaluación
# MAGIC
# MAGIC ## Importante
# MAGIC
# MAGIC Los productos Gold NO representan fraude confirmado.
# MAGIC
# MAGIC Representan:
# MAGIC
# MAGIC - señales de riesgo,
# MAGIC - niveles de riesgo,
# MAGIC - candidatos a investigación,
# MAGIC - indicadores Trust & Safety.
# MAGIC
# MAGIC Un caso solo podría considerarse confirmado mediante
# MAGIC evidencia adicional o revisión humana.


# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "yelp_silver"
GOLD_SCHEMA = "yelp_gold"
GOV_SCHEMA = "yelp_gov"

SILVER = f"{CATALOG}.{SILVER_SCHEMA}"
GOLD = f"{CATALOG}.{GOLD_SCHEMA}"
GOV = f"{CATALOG}.{GOV_SCHEMA}"

PIPELINE_NAME = "E7_04_gold_productos_datos"


print("=" * 72)
print("YELPCONNECT - SILVER -> GOLD")
print("=" * 72)

print(
    f"Pipeline : {PIPELINE_NAME}"
)

print("=" * 72)


# COMMAND ----------

# ============================================================
# 3. CREAR SCHEMA GOLD
# ============================================================

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    {CATALOG}.{GOLD_SCHEMA}
    """
)


print(
    f"✔ Schema disponible: "
    f"{GOLD}"
)


# COMMAND ----------

# ============================================================
# 4. ÚLTIMA EJECUCIÓN T&S EXITOSA
# ============================================================

latest_run = (
    spark.sql(
        f"""
        SELECT
            evaluation_run_id,
            dataset_version,
            started_at,
            finished_at

        FROM
            {GOV}.evaluation_run

        WHERE
            execution_status = 'SUCCESS'

        ORDER BY
            started_at DESC

        LIMIT 1
        """
    )
    .first()
)


if latest_run is None:

    raise Exception(
        """
        ❌ No existe una ejecución
        Trust & Safety exitosa.
        """
    )


EVALUATION_RUN_ID = (
    latest_run.evaluation_run_id
)


print(
    f"✔ Evaluation Run: "
    f"{EVALUATION_RUN_ID}"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 5. Dimensión Business


# COMMAND ----------

# ============================================================
# 5. GOLD DIM BUSINESS
# ============================================================

dim_business = (
    spark.table(
        f"{SILVER}.slv_business"
    )

    .select(

        "business_id",

        F.col(
            "nombre"
        ).alias(
            "business_name"
        ),

        F.col(
            "ciudad"
        ).alias(
            "city"
        ),

        F.col(
            "estado"
        ).alias(
            "state"
        ),

        F.col(
            "estrellas"
        ).alias(
            "business_stars"
        ),

        F.col(
            "n_resenas"
        ).alias(
            "review_count"
        ),

        F.col(
            "esta_abierto"
        ).alias(
            "is_open"
        ),

        F.col(
            "categorias"
        ).alias(
            "categories"
        )
    )

    .withColumn(
        "business_sk",

        F.sha2(
            F.col(
                "business_id"
            ),
            256
        )
    )

    .withColumn(
        "_gold_loaded_at",
        F.current_timestamp()
    )
)


(
    dim_business

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{GOLD}.gld_dim_business"
    )
)


print(
    "✔ gld_dim_business"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 6. Dimensión User


# COMMAND ----------

# ============================================================
# 6. GOLD DIM USER
# ============================================================

dim_user = (
    spark.table(
        f"{SILVER}.slv_user"
    )

    .select(

        "user_id",

        F.col(
            "nombre"
        ).alias(
            "user_name"
        ),

        "yelping_since",

        "review_count",

        "fans",

        "average_stars"
    )

    .withColumn(
        "user_sk",

        F.sha2(
            F.col(
                "user_id"
            ),
            256
        )
    )

    .withColumn(
        "_gold_loaded_at",
        F.current_timestamp()
    )
)


(
    dim_user

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{GOLD}.gld_dim_user"
    )
)


print(
    "✔ gld_dim_user"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 7. Dimensión Rule
# MAGIC
# MAGIC El grano es:
# MAGIC
# MAGIC **1 fila = 1 versión de regla**


# COMMAND ----------

# ============================================================
# 7. GOLD DIM RULE
# ============================================================

dim_rule = (
    spark.table(
        f"{GOV}.evaluation_rule_version"
    )

    .alias("v")

    .join(

        spark.table(
            f"{GOV}.evaluation_rule"
        )
        .alias("r"),

        F.col(
            "v.rule_id"
        )
        ==
        F.col(
            "r.rule_id"
        ),

        "inner"
    )

    .select(

        F.col(
            "v.rule_version_id"
        ),

        F.col(
            "v.rule_id"
        ),

        F.col(
            "r.rule_name"
        ),

        F.col(
            "r.rule_type"
        ),

        F.col(
            "r.business_definition"
        ),

        F.col(
            "v.version"
        ),

        F.col(
            "v.description"
        ),

        F.col(
            "v.parameters_json"
        ),

        F.col(
            "v.threshold"
        ),

        F.col(
            "v.weight"
        ),

        F.col(
            "v.status"
        )
    )

    .withColumn(
        "rule_sk",

        F.sha2(
            F.col(
                "rule_version_id"
            ),
            256
        )
    )
)


(
    dim_rule

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{GOLD}.gld_dim_rule"
    )
)


print(
    "✔ gld_dim_rule"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 8. Dimensión Evaluation Run


# COMMAND ----------

# ============================================================
# 8. GOLD DIM EVALUATION RUN
# ============================================================

dim_run = (
    spark.table(
        f"{GOV}.evaluation_run"
    )

    .withColumn(
        "evaluation_run_sk",

        F.sha2(
            F.col(
                "evaluation_run_id"
            ),
            256
        )
    )
)


(
    dim_run

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{GOLD}.gld_dim_evaluation_run"
    )
)


print(
    "✔ gld_dim_evaluation_run"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 9. Fact Review Evaluation
# MAGIC
# MAGIC Grano:
# MAGIC
# MAGIC **1 Review × 1 versión de regla × 1 ejecución**


# COMMAND ----------

# ============================================================
# 9. FACT REVIEW EVALUATION
# ============================================================

evaluations = (
    spark.table(
        f"{SILVER}.slv_review_evaluation"
    )

    .filter(
        F.col(
            "evaluation_run_id"
        )
        ==
        EVALUATION_RUN_ID
    )
)


review_base = (
    spark.table(
        f"{SILVER}.slv_review"
    )

    .select(
        "review_id",
        "user_id",
        "business_id",
        "review_ts",
        "stars"
    )
)


fact = (
    evaluations

    .join(
        review_base,
        "review_id",
        "left"
    )

    .withColumn(
        "user_sk",

        F.sha2(
            F.col(
                "user_id"
            ),
            256
        )
    )

    .withColumn(
        "business_sk",

        F.sha2(
            F.col(
                "business_id"
            ),
            256
        )
    )

    .withColumn(
        "rule_sk",

        F.sha2(
            F.col(
                "rule_version_id"
            ),
            256
        )
    )

    .withColumn(
        "evaluation_run_sk",

        F.sha2(
            F.col(
                "evaluation_run_id"
            ),
            256
        )
    )

    .withColumn(
        "evaluation_count",
        F.lit(1)
    )

    .select(

        "evaluation_id",

        "review_id",

        "user_sk",

        "business_sk",

        "rule_sk",

        "evaluation_run_sk",

        "evaluation_run_id",

        "rule_id",

        "rule_version_id",

        "evaluation_result",

        "risk_score",

        "reason",

        "stars",

        "review_ts",

        "evaluated_at",

        "evaluation_count"
    )
)


(
    fact

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{GOLD}.gld_fact_review_evaluation"
    )
)


print(
    "✔ gld_fact_review_evaluation"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 10. Producto Review Risk
# MAGIC
# MAGIC Este producto consolida las señales por Review.
# MAGIC
# MAGIC Importante:
# MAGIC
# MAGIC `requires_investigation = true`
# MAGIC
# MAGIC NO significa fraude confirmado.
# MAGIC
# MAGIC Significa que la Review cumple criterios heurísticos
# MAGIC suficientes para priorizar una revisión.


# COMMAND ----------

# ============================================================
# 10. GOLD REVIEW RISK
# ============================================================

summary = (
    spark.table(
        f"{SILVER}.slv_review_signal_summary"
    )

    .filter(
        F.col(
            "evaluation_run_id"
        )
        ==
        EVALUATION_RUN_ID
    )
)


review_detail = (
    spark.table(
        f"{SILVER}.slv_review"
    )

    .select(

        "review_id",
        "user_id",
        "business_id",
        "stars",
        "review_text",
        "review_ts"
    )
)


review_risk = (
    summary

    .join(
        review_detail,
        "review_id",
        "left"
    )

    .join(

        spark.table(
            f"{SILVER}.slv_business"
        )

        .select(
            "business_id",

            F.col(
                "nombre"
            ).alias(
                "business_name"
            ),

            F.col(
                "ciudad"
            ).alias(
                "city"
            )
        ),

        "business_id",

        "left"
    )


    # --------------------------------------------------------
    # Candidato a investigación
    # --------------------------------------------------------

    .withColumn(
        "requires_investigation",

        (
            F.col(
                "signal_count"
            ) >= 2
        )

        |

        (
            F.col(
                "risk_score"
            ) >= 0.35
        )
    )


    # --------------------------------------------------------
    # Prioridad operacional
    # --------------------------------------------------------

    .withColumn(
        "investigation_priority",

        F.when(
            (
                F.col(
                    "signal_count"
                ) >= 3
            )

            |

            (
                F.col(
                    "risk_score"
                ) >= 0.60
            ),

            F.lit(
                "P1"
            )
        )

        .when(
            (
                F.col(
                    "signal_count"
                ) >= 2
            )

            |

            (
                F.col(
                    "risk_score"
                ) >= 0.35
            ),

            F.lit(
                "P2"
            )
        )

        .when(
            F.col(
                "signal_count"
            ) >= 1,

            F.lit(
                "P3"
            )
        )

        .otherwise(
            F.lit(
                "NONE"
            )
        )
    )


    # --------------------------------------------------------
    # Clasificación explicable
    # --------------------------------------------------------

    .withColumn(
        "risk_classification",

        F.when(
            F.col(
                "requires_investigation"
            ),

            F.lit(
                "POTENTIALLY_SUSPICIOUS"
            )
        )

        .when(
            F.col(
                "signal_count"
            ) > 0,

            F.lit(
                "SIGNAL_DETECTED"
            )
        )

        .otherwise(
            F.lit(
                "NO_RELEVANT_SIGNAL"
            )
        )
    )


    .withColumn(
        "_gold_loaded_at",
        F.current_timestamp()
    )
)


(
    review_risk

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{GOLD}.gld_review_risk"
    )
)


print(
    "✔ gld_review_risk"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 11. Cola de investigación
# MAGIC
# MAGIC Este producto contiene únicamente Reviews
# MAGIC priorizadas para análisis Trust & Safety.


# COMMAND ----------

# ============================================================
# 11. INVESTIGATION QUEUE
# ============================================================

investigation_queue = (
    review_risk

    .filter(
        F.col(
            "requires_investigation"
        )
        ==
        True
    )

    .select(

        "review_id",

        "user_id",

        "business_id",

        "business_name",

        "review_ts",

        "stars",

        "review_text",

        "signal_count",

        "signal_rules",

        "risk_score",

        "signal_level",

        "risk_classification",

        "investigation_priority",

        "evaluation_run_id"
    )


    .withColumn(
        "investigation_status",

        F.lit(
            "PENDING_REVIEW"
        )
    )
)


(
    investigation_queue

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{GOLD}.gld_investigation_queue"
    )
)


print(
    "✔ gld_investigation_queue"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 12. KPI Trust & Safety por Business


# COMMAND ----------

# ============================================================
# 12. KPI BUSINESS
# ============================================================

kpi_business = (
    review_risk

    .groupBy(
        "business_id",
        "business_name",
        "city"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "reviews_evaluated"
        ),

        F.sum(
            F.when(
                F.col(
                    "signal_count"
                ) > 0,
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "reviews_with_signal"
        ),

        F.sum(
            F.when(
                F.col(
                    "requires_investigation"
                ),
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "reviews_for_investigation"
        ),

        F.avg(
            "risk_score"
        ).alias(
            "avg_risk_score"
        ),

        F.max(
            "risk_score"
        ).alias(
            "max_risk_score"
        )
    )


    .withColumn(
        "signal_rate_pct",

        F.round(
            (
                F.col(
                    "reviews_with_signal"
                )
                /
                F.col(
                    "reviews_evaluated"
                )
            )
            *
            100,
            2
        )
    )


    .withColumn(
        "investigation_rate_pct",

        F.round(
            (
                F.col(
                    "reviews_for_investigation"
                )
                /
                F.col(
                    "reviews_evaluated"
                )
            )
            *
            100,
            2
        )
    )
)


(
    kpi_business

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{GOLD}.gld_kpi_trust_safety_business"
    )
)


print(
    "✔ gld_kpi_trust_safety_business"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 13. KPI diario


# COMMAND ----------

# ============================================================
# 13. KPI DAILY
# ============================================================

kpi_daily = (
    review_risk

    .withColumn(
        "review_date",

        F.to_date(
            "review_ts"
        )
    )

    .groupBy(
        "review_date"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "reviews_evaluated"
        ),

        F.sum(
            F.when(
                F.col(
                    "signal_count"
                ) > 0,
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "reviews_with_signal"
        ),

        F.sum(
            F.when(
                F.col(
                    "requires_investigation"
                ),
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "reviews_for_investigation"
        ),

        F.avg(
            "risk_score"
        ).alias(
            "avg_risk_score"
        )
    )


    .withColumn(
        "signal_rate_pct",

        F.round(
            (
                F.col(
                    "reviews_with_signal"
                )
                /
                F.col(
                    "reviews_evaluated"
                )
            )
            *
            100,
            2
        )
    )
)


(
    kpi_daily

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        f"{GOLD}.gld_kpi_trust_safety_daily"
    )
)


print(
    "✔ gld_kpi_trust_safety_daily"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 14. Validación Gold


# COMMAND ----------

# ============================================================
# 14. VALIDACIONES
# ============================================================

gold_tables = [

    "gld_dim_business",
    "gld_dim_user",
    "gld_dim_rule",
    "gld_dim_evaluation_run",
    "gld_fact_review_evaluation",
    "gld_review_risk",
    "gld_investigation_queue",
    "gld_kpi_trust_safety_business",
    "gld_kpi_trust_safety_daily"
]


validation = []


for table_name in gold_tables:

    full_name = (
        f"{GOLD}.{table_name}"
    )

    count = (
        spark.table(
            full_name
        )
        .count()
    )

    validation.append(
        (
            table_name,
            count
        )
    )


display(
    spark.createDataFrame(
        validation,
        [
            "gold_product",
            "rows"
        ]
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 15. Distribución Trust & Safety


# COMMAND ----------

display(
    spark.sql(
        f"""
        SELECT

            risk_classification,

            investigation_priority,

            COUNT(*) AS reviews

        FROM
            {GOLD}.gld_review_risk

        GROUP BY
            risk_classification,
            investigation_priority

        ORDER BY
            investigation_priority,
            risk_classification
        """
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 16. Top candidatos para investigación


# COMMAND ----------

display(
    spark.sql(
        f"""
        SELECT

            review_id,

            business_name,

            stars,

            signal_count,

            signal_rules,

            risk_score,

            signal_level,

            investigation_priority

        FROM
            {GOLD}.gld_investigation_queue

        ORDER BY

            CASE investigation_priority

                WHEN 'P1'
                    THEN 1

                WHEN 'P2'
                    THEN 2

                ELSE 3

            END,

            risk_score DESC,

            signal_count DESC

        LIMIT 100
        """
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # Definition of Done — Gold
# MAGIC
# MAGIC - [ ] Dimensiones Gold creadas.
# MAGIC - [ ] Fact Review Evaluation creada.
# MAGIC - [ ] Producto consolidado de riesgo creado.
# MAGIC - [ ] Cola de investigación creada.
# MAGIC - [ ] KPIs por Business creados.
# MAGIC - [ ] KPIs diarios creados.
# MAGIC - [ ] Las señales son explicables mediante `signal_rules`.
# MAGIC - [ ] Cada resultado mantiene `evaluation_run_id`.
# MAGIC - [ ] No se etiqueta fraude confirmado sin ground truth.


# COMMAND ----------

print()
print("=" * 72)
print("GOLD TRUST & SAFETY FINALIZADO")
print("=" * 72)

print(
    f"Evaluation Run : "
    f"{EVALUATION_RUN_ID}"
)

print(
    "✔ Productos Gold publicados."
)

print("=" * 72)